In [77]:
import os

import ROOT


def compare(files, objNames, legendTexts, outputPath, suffix):
    colors = [
        # ROOT.kBlack,       
        ROOT.kRed-4,       
        ROOT.kBlue-4, 
        ROOT.kGreen+2,     
        ROOT.kOrange+7,   
        # ROOT.kYellow-7,    
        ROOT.kMagenta-3,   
        ROOT.kCyan-3,      
        ROOT.kSpring-5,    
        ROOT.kViolet-4,    
        ROOT.kTeal-5,    
        ROOT.kGray+1    
    ]
    markerStyles = [
        20,
        20,
        20,
        20,
        20,
        21,
        21,
        21,
        21,
        22,
    ]

    canvas = ROOT.TCanvas("c1", "c1", 2400, 1800)
    canvas.SetLeftMargin(0.12)
    canvas.SetBottomMargin(0.15)
    canvas.SetRightMargin(0.10)
    canvas.SetTopMargin(0.08)
    Legend = ROOT.TLegend(0.2, 0.7, 0.6, 0.9)
    Legend.SetBorderSize(0)
    Legend.SetFillStyle(0)
    legendFont = 42
    Legend.SetTextFont(legendFont)  
    Legend.SetTextSize(0.04)
    ROOT.gStyle.SetOptStat(0)
    # frame = canvas.DrawFrame(-1.58, 2200, 4.72, 12000000)
    # frame.GetYaxis().SetLogY
    # frame.GetYaxis().SetTitle("#frac{dN}{d#Delta#varphi}")
    # frame.GetXaxis().SetTitle("#Delta#varphi (rad)")

    objs = []
    new_objs = []
    ratio_objs = []
    minY, maxY = 0, 0
    for i, fpath, objName, color, markerStyle in zip(range(len(files)), files, objNames, colors, markerStyles):
        f = ROOT.TFile.Open(fpath, "read")
        obj = f.Get(objName)
        if isinstance(obj, ROOT.TH2):
            obj = obj.ProjectionY(f"proj_{i}")
            # obj.Scale(1/obj.Integral())
            temp_min = obj.GetMinimum()
            temp = obj.Clone(f"temp_{i}")
            temp.Reset()
            for b in range(1, obj.GetNbinsX() + 1):
                temp.SetBinContent(b, -temp_min)
            obj.Add(temp)
        obj.SetDirectory(0) if hasattr(obj, "SetDirectory") else None
        obj.SetLineColor(color)
        obj.SetLineWidth(3)
        obj.SetMarkerColor(color)
        obj.SetMarkerStyle(markerStyle)
        obj.SetMarkerSize(1)
        if files.index(fpath) == 0:
            # obj.SetTitle("Inclusive D^{0} v_{2} in OO collisions")
            # obj.Scale(1/0.07)
            obj.GetYaxis().SetTitle("#frac{dN}{d#Delta#varphi}")
            obj.GetYaxis().SetRangeUser(0, obj.GetMaximum()*1.5)
            obj.GetXaxis().SetTitle("#Delta#varphi (rad)")
            obj.GetXaxis().SetTitleOffset(1.2)
            obj.Draw("same pe")
        elif files.index(fpath) == len(files)-1:
            obj.SetLineColor(ROOT.kBlack)
            obj.SetLineWidth(3)
            obj.SetMarkerColor(ROOT.kBlack)
            obj.SetMarkerStyle(20)
            obj.SetMarkerSize(1)
            obj.Draw("same pe")
        else:
            # obj.SetTitle("")
            # obj.GetYaxis().SetTitle("inclusive D^{0} v_{2}")
            # obj.GetXaxis().SetTitle("p_{T} (GeV/c)")
            obj.Draw("same pe")
        Legend.AddEntry(obj, f"{legendTexts[i]} - {temp_min}", "lp")
        objs.append(obj)
        new_objs.append(obj)
        f.Close()
    Legend.Draw()
    # canvas.Draw()
    canvas.Update()
    # the last one is the demestor, and the rest are the numerator, calculate and draw the ratio

    canvas_ratio = ROOT.TCanvas("c2", "c2", 2400, 1800)
    canvas_ratio.SetLeftMargin(0.12)
    canvas_ratio.SetBottomMargin(0.15)
    canvas_ratio.SetRightMargin(0.10)
    canvas_ratio.SetTopMargin(0.08)
    # frame_ratio = canvas_ratio.DrawFrame(0, 0, 12, 2.1)
    maxs = [obj.GetMaximum() for obj in new_objs[:-1]]
    mins = [obj.GetMinimum() for obj in new_objs[:-1]]
    max_ratio = max(maxs) / min(mins) if min(mins) != 0 else 2
    min_ratio = min(mins) / max(maxs) if max(maxs) != 0 else 0
    frame_ratio = canvas_ratio.DrawFrame(-1.58, min_ratio, 4.72, 10)

    for i in range(len(new_objs)-1):
        ratio = new_objs[i].Clone(f"ratio_{i}")
        ratio.Divide(new_objs[-1])
        # for b in range(1, ratio.GetNbinsX() + 1):
        #     num = new_objs[i].GetBinContent(b)
        #     den = new_objs[-1].GetBinContent(b)
            
        #     if den != 0:
        #         temp_ratio = num / den
        #         ratio.SetBinContent(b, num / den)
        #         # num_err = new_objs[i].GetBinError(b)
        #         # den_err = new_objs[-1].GetBinError(b)
        #         # ratio_err = temp_ratio * ((num_err / num) ** 2 + (den_err / den) ** 2) ** 0.5 if num != 0 else 0
        #         # ratio.SetBinError(b, ratio_err)
        #     else:
        #         ratio.SetBinContent(b, 0)
        #         ratio.SetBinError(b, 0)
        ratio.SetLineColor(colors[i])
        ratio.SetLineWidth(3)
        ratio.SetMarkerColor(colors[i])
        ratio.SetMarkerStyle(markerStyles[i])
        ratio.SetMarkerSize(1)
        # ratio.GetYaxis().SetTitle("Ratio to Biao sp prompt")
        ratio.GetXaxis().SetTitle("#Delta#varphi (rad)")
        ratio.Draw("same")
        ratio_objs.append(ratio)
    # line = ROOT.TLine(0, 1, 12, 1)
    # line.SetLineStyle(2)
    # line.SetLineColor(ROOT.kBlack)
    # line.Draw("same")
    # canvas_ratio.Draw()
    canvas_ratio.Update()

    os.system(f"mkdir -p {outputPath}")
    ouput = outputPath + f"compare_{suffix}.root"
    canvas.SaveAs(outputPath + f"compare_{suffix}.png")
    canvas_ratio.SaveAs(outputPath + f"compare_ratio_{suffix}.png")
    outputfile = ROOT.TFile(ouput, "recreate")
    canvas.Write("c1")
    canvas_ratio.Write("c2")
    for obj in objs:
        obj.Write()
    for obj in ratio_objs:
        obj.Write()
    outputfile.Close()



In [ ]:
outputPath = "/home/wuct/MetaData/DATA/OO/apass2/corr/results/fifth/k020/etavariation/compare/"
files = [

]
files = [
    "/home/wuct/MetaData/DATA/OO/apass2/corr/results/fifth/k020/etavariation/CorrelExtract_0d_1d3_AppDeltaPhi/CorrelationsResults/CorrelationsResults.root",
    "/home/wuct/MetaData/DATA/OO/apass2/corr/results/fifth/k020/etavariation/CorrelExtract_0d2_1d3_AppDeltaPhi/CorrelationsResults/CorrelationsResults.root",
    "/home/wuct/MetaData/DATA/OO/apass2/corr/results/fifth/k020/etavariation/CorrelExtract_0d4_1d3_AppDeltaPhi/CorrelationsResults/CorrelationsResults.root",
    "/home/wuct/MetaData/DATA/OO/apass2/corr/results/fifth/k020/fitprocedure/CorrelExtract_0d8_1d3_AppDeltaPhi/CorrelationsResults/CorrelationsResults.root",
]
objNames = [
    "PtCandBin_20_25/PtHadBin_2_30/DeltaPhiBin_0_392/hCorrel_ME_2D",
    "PtCandBin_20_25/PtHadBin_2_30/DeltaPhiBin_0_392/hCorrel_ME_2D",
    "PtCandBin_20_25/PtHadBin_2_30/DeltaPhiBin_0_392/hCorrel_ME_2D",
    "PtCandBin_20_25/PtHadBin_2_30/DeltaPhiBin_0_392/hCorrel_ME_2D"
]
legendTexts = [
    "ME w eta gap 0.0-1.3",
    "ME w eta gap 0.2-1.3",
    "ME w eta gap 0.4-1.3",
    "ME w eta gap 0.8-1.3",
]

suffix = "ME_AppDeltaPhi"
compare(files, objNames, legendTexts, outputPath, suffix)

# suffix = "0d8_1d3_roofit_deltaPhi_wrtNF"
# compare(files, objNames, legendTexts, outputPath, suffix)
# suffix = "0d8_1d3_roofit_wPed_freeLM"
# compare([files[0], files[4], files[7]], [objNames[0], objNames[4], objNames[7]], [legendTexts[0], legendTexts[4], legendTexts[7]], outputPath, suffix)

# suffix = "0d8_1d3_roofit_woPed_freeLM"
# compare([files[1], files[5], files[7]], [objNames[1], objNames[5], objNames[7]], [legendTexts[1], legendTexts[5], legendTexts[7]], outputPath, suffix)

# suffix = "0d8_1d3_roofit_woNFSub"
# compare([files[3], files[6], files[7]], [objNames[3], objNames[6], objNames[7]], [legendTexts[3], legendTexts[6], legendTexts[7]], outputPath, suffix)

Info in <TCanvas::Print>: png file /home/wuct/MetaData/DATA/OO/apass2/corr/results/fifth/k020/etavariation/compare/compare_ME_AppDeltaPhi.png has been created
Info in <TCanvas::Print>: png file /home/wuct/MetaData/DATA/OO/apass2/corr/results/fifth/k020/etavariation/compare/compare_ratio_ME_AppDeltaPhi.png has been created


In [82]:
import os

import ROOT


def compare_se(files, objNames, legendTexts, outputPath, suffix):
    colors = [
        # ROOT.kBlack,       
        ROOT.kRed-4,       
        ROOT.kBlue-4, 
        ROOT.kGreen+2,     
        ROOT.kOrange+7,   
        # ROOT.kYellow-7,    
        ROOT.kMagenta-3,   
        ROOT.kCyan-3,      
        ROOT.kSpring-5,    
        ROOT.kViolet-4,    
        ROOT.kTeal-5,    
        ROOT.kGray+1    
    ]
    markerStyles = [
        20,
        20,
        20,
        20,
        20,
        21,
        21,
        21,
        21,
        22,
    ]

    canvas = ROOT.TCanvas("c1", "c1", 2400, 1800)
    canvas.SetLeftMargin(0.15)
    canvas.SetBottomMargin(0.15)
    canvas.SetRightMargin(0.05)
    canvas.SetTopMargin(0.08)
    Legend = ROOT.TLegend(0.2, 0.7, 0.6, 0.9)
    Legend.SetBorderSize(0)
    Legend.SetFillStyle(0)
    legendFont = 42
    Legend.SetTextFont(legendFont)  
    Legend.SetTextSize(0.04)
    ROOT.gStyle.SetOptStat(0)
    frame = canvas.DrawFrame(-1.58, 0, 4.72, 5)
    # frame.GetYaxis().SetLogY
    # frame.GetYaxis().SetTitle("#frac{dN}{d#Delta#varphi}")
    # frame.GetXaxis().SetTitle("#Delta#varphi (rad)")

    objs = []
    new_objs = []
    ratio_objs = []
    minY, maxY = 0, 0
    for i, fpath, objName, color, markerStyle in zip(range(len(files)), files, objNames, colors, markerStyles):
        f = ROOT.TFile.Open(fpath, "read")
        obj = f.Get(objName)
        if isinstance(obj, ROOT.TH2):
            obj = obj.ProjectionY(f"proj_{i}")
        elif isinstance(obj, ROOT.TH1):
            # obj.Scale(1/obj.Integral())
            temp_min = obj.GetMinimum()
            temp = obj.Clone(f"temp_{i}")
            temp.Reset()
            for b in range(1, obj.GetNbinsX() + 1):
                temp.SetBinContent(b, -temp_min)
            # obj.Add(temp)
        else:
            print(f"Object {objName} in file {fpath} is neither TH1 nor TH2. Skipping.")
            continue
        obj.SetDirectory(0) if hasattr(obj, "SetDirectory") else None
        obj.SetLineColor(color)
        obj.SetLineWidth(3)
        obj.SetMarkerColor(color)
        obj.SetMarkerStyle(markerStyle)
        obj.SetMarkerSize(1)
        if files.index(fpath) == 0:
            # obj.SetTitle("Inclusive D^{0} v_{2} in OO collisions")
            # obj.Scale(1/0.07)
            obj.GetYaxis().SetTitle("1/dN^{trigger} #frac{dN}{d#Delta#varphi}")
            obj.GetYaxis().SetRangeUser(0, obj.GetMaximum()*1.5)
            obj.GetXaxis().SetTitle("#Delta#varphi (rad)")
            obj.GetXaxis().SetTitleOffset(1.2)
            obj.Draw("same pe")
        elif files.index(fpath) == len(files)-1:
            obj.SetLineColor(ROOT.kBlack)
            obj.SetLineWidth(3)
            obj.SetMarkerColor(ROOT.kBlack)
            obj.SetMarkerStyle(20)
            obj.SetMarkerSize(1)
            obj.Draw("same pe")
        else:
            # obj.SetTitle("")
            # obj.GetYaxis().SetTitle("inclusive D^{0} v_{2}")
            # obj.GetXaxis().SetTitle("p_{T} (GeV/c)")
            obj.Draw("same pe")
        Legend.AddEntry(obj, f"{legendTexts[i]} - {temp_min:.2f}", "lp")
        objs.append(obj)
        new_objs.append(obj)
        f.Close()
    Legend.Draw()
    # canvas.Draw()
    canvas.Update()
    # the last one is the demestor, and the rest are the numerator, calculate and draw the ratio

    canvas_ratio = ROOT.TCanvas("c2", "c2", 2400, 1800)
    canvas_ratio.SetLeftMargin(0.12)
    canvas_ratio.SetBottomMargin(0.15)
    canvas_ratio.SetRightMargin(0.10)
    canvas_ratio.SetTopMargin(0.08)
    # frame_ratio = canvas_ratio.DrawFrame(0, 0, 12, 2.1)
    maxs = [obj.GetMaximum() for obj in new_objs[:-1]]
    mins = [obj.GetMinimum() for obj in new_objs[:-1]]
    max_ratio = max(maxs) / min(mins) if min(mins) != 0 else 2
    min_ratio = min(mins) / max(maxs) if max(maxs) != 0 else 0
    frame_ratio = canvas_ratio.DrawFrame(-1.58, min_ratio, 4.72, 10)

    for i in range(len(new_objs)-1):
        ratio = new_objs[i].Clone(f"ratio_{i}")
        ratio.Divide(new_objs[-1])
        # for b in range(1, ratio.GetNbinsX() + 1):
        #     num = new_objs[i].GetBinContent(b)
        #     den = new_objs[-1].GetBinContent(b)
            
        #     if den != 0:
        #         temp_ratio = num / den
        #         ratio.SetBinContent(b, num / den)
        #         # num_err = new_objs[i].GetBinError(b)
        #         # den_err = new_objs[-1].GetBinError(b)
        #         # ratio_err = temp_ratio * ((num_err / num) ** 2 + (den_err / den) ** 2) ** 0.5 if num != 0 else 0
        #         # ratio.SetBinError(b, ratio_err)
        #     else:
        #         ratio.SetBinContent(b, 0)
        #         ratio.SetBinError(b, 0)
        ratio.SetLineColor(colors[i])
        ratio.SetLineWidth(3)
        ratio.SetMarkerColor(colors[i])
        ratio.SetMarkerStyle(markerStyles[i])
        ratio.SetMarkerSize(1)
        # ratio.GetYaxis().SetTitle("Ratio to Biao sp prompt")
        ratio.GetXaxis().SetTitle("#Delta#varphi (rad)")
        ratio.Draw("same")
        ratio_objs.append(ratio)
    # line = ROOT.TLine(0, 1, 12, 1)
    # line.SetLineStyle(2)
    # line.SetLineColor(ROOT.kBlack)
    # line.Draw("same")
    # canvas_ratio.Draw()
    canvas_ratio.Update()

    os.system(f"mkdir -p {outputPath}")
    ouput = outputPath + f"compare_{suffix}.root"
    canvas.SaveAs(outputPath + f"compare_{suffix}.png")
    canvas_ratio.SaveAs(outputPath + f"compare_ratio_{suffix}.png")
    outputfile = ROOT.TFile(ouput, "recreate")
    canvas.Write("c1")
    canvas_ratio.Write("c2")
    for obj in objs:
        obj.Write()
    for obj in ratio_objs:
        obj.Write()
    outputfile.Close()



In [83]:
outputPath = "/home/wuct/MetaData/DATA/OO/apass2/corr/results/fifth/k020/etavariation/compare/A"
files = [

]
files = [
    "/home/wuct/MetaData/DATA/OO/apass2/corr/results/fifth/k60100/etavariation/CorrelExtract_0d_1d3_AppDeltaPhi/CorrelationsResults/CorrelationsResults.root",
    "/home/wuct/MetaData/DATA/OO/apass2/corr/results/fifth/k60100/etavariation/CorrelExtract_0d_1d3_Appmass/CorrelationsResults/CorrelationsResults.root",
    # "/home/wuct/MetaData/DATA/OO/apass2/corr/results/fifth/k60100/etavariation/CorrelExtract_0d4_1d3_AppDeltaPhi/CorrelationsResults/CorrelationsResults.root",
    # "/home/wuct/MetaData/DATA/OO/apass2/corr/results/fifth/k60100/fitprocedure/CorrelExtract_0d8_1d3_AppDeltaPhi/CorrelationsResults/CorrelationsResults.root",
]
objNames = [
    "PtCandBin_50_60/PtHadBin_2_30/DeltaPhiBin_0_392/hNormalizedCorrectedCorrel",
    "PtCandBin_50_60/PtHadBin_2_30/InvMassBin_1720_1760/hNormalizedCorrectedCorrel",
    # "PtCandBin_20_25/PtHadBin_2_30/DeltaPhiBin_0_392/hNormalizedCorrectedCorrel",
    # "PtCandBin_20_25/PtHadBin_2_30/DeltaPhiBin_0_392/hNormalizedCorrectedCorrel"
]
legendTexts = [
    "Corrected SE w eta gap 0.0-1.3 LM",
    "Corrected SE w eta gap 0.0-1.3 LM mass",
    # "Corrected SE w eta gap 0.4-1.3 LM",
    # "Corrected SE w eta gap 0.8-1.3 LM",
]

suffix = "CorrectedSE_AppDeltaPhi_LM"
compare_se(files, objNames, legendTexts, outputPath, suffix)


Error in <TH1D::Divide>: Histograms have different number of bins
Info in <TCanvas::Print>: png file /home/wuct/MetaData/DATA/OO/apass2/corr/results/fifth/k020/etavariation/compare/Acompare_CorrectedSE_AppDeltaPhi_LM.png has been created
Info in <TCanvas::Print>: png file /home/wuct/MetaData/DATA/OO/apass2/corr/results/fifth/k020/etavariation/compare/Acompare_ratio_CorrectedSE_AppDeltaPhi_LM.png has been created


In [46]:
outputPath = "/home/wuct/MetaData/DATA/OO/apass2/corr/results/fifth/k020/etavariation/compare/"
files = [

]
files = [
    "/home/wuct/MetaData/DATA/OO/apass2/corr/results/fifth/k020/etavariation/CorrelExtract_0d_1d3_AppDeltaPhi/CorrelationsResults/CorrelationsResults.root",
    "/home/wuct/MetaData/DATA/OO/apass2/corr/results/fifth/k020/etavariation/CorrelExtract_0d2_1d3_AppDeltaPhi/CorrelationsResults/CorrelationsResults.root",
    "/home/wuct/MetaData/DATA/OO/apass2/corr/results/fifth/k020/etavariation/CorrelExtract_0d4_1d3_AppDeltaPhi/CorrelationsResults/CorrelationsResults.root",
    "/home/wuct/MetaData/DATA/OO/apass2/corr/results/fifth/k020/fitprocedure/CorrelExtract_0d8_1d3_AppDeltaPhi/CorrelationsResults/CorrelationsResults.root",
]
objNames = [
    "PtCandBin_50_60/PtHadBin_2_30/DeltaPhiBin_0_392/hNormalizedCorrectedCorrel",
    "PtCandBin_50_60/PtHadBin_2_30/DeltaPhiBin_0_392/hNormalizedCorrectedCorrel",
    "PtCandBin_50_60/PtHadBin_2_30/DeltaPhiBin_0_392/hNormalizedCorrectedCorrel",
    "PtCandBin_50_60/PtHadBin_2_30/DeltaPhiBin_0_392/hNormalizedCorrectedCorrel"
]
legendTexts = [
    "Corrected SE w eta gap 0.0-1.3",
    "Corrected SE w eta gap 0.2-1.3",
    "Corrected SE w eta gap 0.4-1.3",
    "Corrected SE w eta gap 0.8-1.3",
]

suffix = "CorrectedSE_AppDeltaPhi_Pt50_60"
compare_se(files, objNames, legendTexts, outputPath, suffix)


Info in <TCanvas::Print>: png file /home/wuct/MetaData/DATA/OO/apass2/corr/results/fifth/k020/etavariation/compare/compare_CorrectedSE_AppDeltaPhi_Pt50_60.png has been created
Info in <TCanvas::Print>: png file /home/wuct/MetaData/DATA/OO/apass2/corr/results/fifth/k020/etavariation/compare/compare_ratio_CorrectedSE_AppDeltaPhi_Pt50_60.png has been created


In [ ]:
import os
import ROOT

def compare_se_combined(files_left, objNames_left, legendTexts_left, 
                        files_right, objNames_right, legendTexts_right, 
                        outputPath, suffix, pt_range=""):
    colors = [
        ROOT.kRed-4, ROOT.kBlue-4, ROOT.kGreen+2, ROOT.kOrange+7,
        ROOT.kMagenta-3, ROOT.kCyan-3, ROOT.kSpring-5, ROOT.kViolet-4,
        ROOT.kTeal-5, ROOT.kGray+1
    ]
    markerStyles = [20, 20, 20, 20, 20, 21, 21, 21, 21, 22]

    # 辅函数：负责读取文件、处理直方图并设置样式
    def process_histograms(files, objNames, pad_id):
        objs = []
        for i, (fpath, objName, color, markerStyle) in enumerate(zip(files, objNames, colors, markerStyles)):
            f = ROOT.TFile.Open(fpath, "read")
            obj = f.Get(objName)
            
            if isinstance(obj, ROOT.TH2):
                obj = obj.ProjectionY(f"proj_{pad_id}_{i}")
                temp_min = obj.GetMinimum()
            elif isinstance(obj, ROOT.TH1):
                temp_min = obj.GetMinimum()
                temp = obj.Clone(f"temp_{pad_id}_{i}")
                temp.Reset()
                for b in range(1, obj.GetNbinsX() + 1):
                    temp.SetBinContent(b, -temp_min)
                # obj.Add(temp)
            else:
                print(f"Object {objName} in file {fpath} is neither TH1 nor TH2. Skipping.")
                continue
                
            obj.SetDirectory(0) if hasattr(obj, "SetDirectory") else None
            obj.SetLineColor(color)
            obj.SetLineWidth(3)
            obj.SetMarkerColor(color)
            obj.SetMarkerStyle(markerStyle)
            obj.SetMarkerSize(1)
            
            # 设置最后一个直方图为黑色
            if i == len(files) - 1:
                obj.SetLineColor(ROOT.kBlack)
                obj.SetMarkerColor(ROOT.kBlack)
                obj.SetMarkerStyle(20)
                
            obj.temp_min = temp_min # 保存平移的数值供Legend使用
            objs.append(obj)
            f.Close()
        return objs

    # 获取左右两组的直方图对象
    objs_left = process_histograms(files_left, objNames_left, "left")
    objs_right = process_histograms(files_right, objNames_right, "right")

    # 计算全局最大Y值以统一高度
    max_y_left = max([obj.GetMaximum() for obj in objs_left]) if objs_left else 0
    max_y_right = max([obj.GetMaximum() for obj in objs_right]) if objs_right else 0
    global_max_y = max(max_y_left, max_y_right) * 1.5

    # ================= 画主图 =================
    # 将画布增宽以适应左右两个子图
    canvas = ROOT.TCanvas("c1", "c1", 3600, 1500) 
    canvas.Divide(2, 1)

    # 画左子图 (Pad 1)
    canvas.cd(1)
    ROOT.gPad.SetLeftMargin(0.15)
    ROOT.gPad.SetBottomMargin(0.15)
    ROOT.gPad.SetRightMargin(0.05)
    ROOT.gPad.SetTopMargin(0.08)
    ROOT.gStyle.SetOptStat(0)

    Legend_left = ROOT.TLegend(0.2, 0.7, 0.6, 0.9)
    Legend_left.SetBorderSize(0)
    Legend_left.SetFillStyle(0)
    Legend_left.SetTextFont(42)  
    Legend_left.SetTextSize(0.04)

    for i, obj in enumerate(objs_left):
        if i == 0:
            obj.GetYaxis().SetTitle("1/dN^{trigger} #frac{dN}{d#Delta#varphi}")
            obj.GetYaxis().SetRangeUser(0, global_max_y) # 统一Y轴
            obj.GetXaxis().SetTitle("#Delta#varphi (rad)")
            obj.GetXaxis().SetTitleOffset(1.2)
            obj.SetTitle(pt_range)
            obj.Draw("same pe")
        else:
            obj.Draw("same pe")
        # Legend_left.AddEntry(obj, f"{legendTexts_left[i]} - {obj.temp_min:.2f}", "lp")
        Legend_left.AddEntry(obj, f"{legendTexts_left[i]}", "lp")
    Legend_left.Draw()

    # 画右子图 (Pad 2)
    canvas.cd(2)
    ROOT.gPad.SetLeftMargin(0.15)
    ROOT.gPad.SetBottomMargin(0.15)
    ROOT.gPad.SetRightMargin(0.05)
    ROOT.gPad.SetTopMargin(0.08)

    Legend_right = ROOT.TLegend(0.2, 0.7, 0.6, 0.9)
    Legend_right.SetBorderSize(0)
    Legend_right.SetFillStyle(0)
    Legend_right.SetTextFont(42)  
    Legend_right.SetTextSize(0.04)

    for i, obj in enumerate(objs_right):
        if i == 0:
            obj.GetYaxis().SetTitle("1/dN^{trigger} #frac{dN}{d#Delta#varphi}")
            obj.GetYaxis().SetRangeUser(0, global_max_y) # 统一Y轴
            obj.GetXaxis().SetTitle("#Delta#varphi (rad)")
            obj.GetXaxis().SetTitleOffset(1.2)
            obj.SetTitle(pt_range)
            obj.Draw("same pe")
        else:
            obj.Draw("same pe")
        # Legend_right.AddEntry(obj, f"{legendTexts_right[i]} - {obj.temp_min:.2f}", "lp")
        Legend_right.AddEntry(obj, f"{legendTexts_right[i]}", "lp")
    Legend_right.Draw()

    canvas.Update()

    # ================= 画Ratio图 =================
    canvas_ratio = ROOT.TCanvas("c2", "c2", 3600, 1500)
    canvas_ratio.Divide(2, 1)

    def process_ratios(objs, pad_id):
        ratio_objs = []
        if not objs: return ratio_objs
        denominator = objs[-1]
        for i in range(len(objs)-1):
            ratio = objs[i].Clone(f"ratio_{pad_id}_{i}")
            ratio.Divide(denominator)
            ratio.SetLineColor(colors[i])
            ratio.SetLineWidth(3)
            ratio.SetMarkerColor(colors[i])
            ratio.SetMarkerStyle(markerStyles[i])
            ratio.SetMarkerSize(1)
            ratio.GetXaxis().SetTitle("#Delta#varphi (rad)")
            ratio_objs.append(ratio)
        return ratio_objs

    ratio_objs_left = process_ratios(objs_left, "left")
    ratio_objs_right = process_ratios(objs_right, "right")

    # 计算全局Ratio Y轴下限
    def get_min_ratio(objs):
        if not objs or len(objs) < 2: return 0
        maxs = [obj.GetMaximum() for obj in objs[:-1]]
        mins = [obj.GetMinimum() for obj in objs[:-1]]
        return min(mins) / max(maxs) if max(maxs) != 0 else 0
        
    global_min_ratio = min(get_min_ratio(objs_left), get_min_ratio(objs_right))

    # 画左Ratio子图
    canvas_ratio.cd(1)
    ROOT.gPad.SetLeftMargin(0.12)
    ROOT.gPad.SetBottomMargin(0.15)
    ROOT.gPad.SetRightMargin(0.10)
    ROOT.gPad.SetTopMargin(0.08)
    frame_ratio_left = ROOT.gPad.DrawFrame(-1.58, global_min_ratio, 4.72, 10)
    for ratio in ratio_objs_left:
        ratio.Draw("same")

    # 画右Ratio子图
    canvas_ratio.cd(2)
    ROOT.gPad.SetLeftMargin(0.12)
    ROOT.gPad.SetBottomMargin(0.15)
    ROOT.gPad.SetRightMargin(0.10)
    ROOT.gPad.SetTopMargin(0.08)
    frame_ratio_right = ROOT.gPad.DrawFrame(-1.58, global_min_ratio, 4.72, 10)
    for ratio in ratio_objs_right:
        ratio.Draw("same")

    canvas_ratio.Update()

    # ================= 保存文件 =================
    os.system(f"mkdir -p {outputPath}")
    output_file_path = os.path.join(outputPath, f"compare_{suffix}.root")
    canvas.SaveAs(os.path.join(outputPath, f"compare_{suffix}.png"))
    canvas_ratio.SaveAs(os.path.join(outputPath, f"compare_ratio_{suffix}.png"))
    
    outputfile = ROOT.TFile(output_file_path, "recreate")
    canvas.Write("c1")
    canvas_ratio.Write("c2")
    for obj in objs_left + objs_right:
        obj.Write()
    for obj in ratio_objs_left + ratio_objs_right:
        obj.Write()
    outputfile.Close()

In [ ]:
outputPath = "/home/wuct/MetaData/DATA/OO/apass2/corr/results/fifth/k020/etavariation/compare/CorrectedSE/k7080/"

# ==================== 左图数据 (对应原本的第三个cell k020) ====================
files_left = [
    "/home/wuct/MetaData/DATA/OO/apass2/corr/results/fifth/k70100/k70100/CorrelExtract_0d_1d3_AppDeltaPhi/CorrelationsResults/CorrelationsResults.root",
    "/home/wuct/MetaData/DATA/OO/apass2/corr/results/fifth/k70100/k70100/CorrelExtract_0d2_1d3_AppDeltaPhi/CorrelationsResults/CorrelationsResults.root",
    "/home/wuct/MetaData/DATA/OO/apass2/corr/results/fifth/k70100/k70100/CorrelExtract_0d4_1d3_AppDeltaPhi/CorrelationsResults/CorrelationsResults.root",
    "/home/wuct/MetaData/DATA/OO/apass2/corr/results/fifth/k70100/k70100/CorrelExtract_0d6_1d3_AppDeltaPhi/CorrelationsResults/CorrelationsResults.root",
    "/home/wuct/MetaData/DATA/OO/apass2/corr/results/fifth/k70100/k70100/CorrelExtract_0d8_1d3_AppDeltaPhi/CorrelationsResults/CorrelationsResults.root",
]
objNames_left = [
    "PtCandBin_50_60/PtHadBin_2_30/DeltaPhiBin_0_392/hNormalizedCorrectedCorrel",
    "PtCandBin_50_60/PtHadBin_2_30/DeltaPhiBin_0_392/hNormalizedCorrectedCorrel",
    "PtCandBin_50_60/PtHadBin_2_30/DeltaPhiBin_0_392/hNormalizedCorrectedCorrel",
    "PtCandBin_50_60/PtHadBin_2_30/DeltaPhiBin_0_392/hNormalizedCorrectedCorrel",
    "PtCandBin_50_60/PtHadBin_2_30/DeltaPhiBin_0_392/hNormalizedCorrectedCorrel"
]
legendTexts_left = [
    "Corrected SE 70-100 0.0-1.3 LM",
    "Corrected SE 70-100 0.2-1.3 LM",
    "Corrected SE 70-100 0.4-1.3 LM",
    "Corrected SE 70-100 0.6-1.3 LM",
    "Corrected SE 70-100 0.8-1.3 LM",
]

# ==================== 右图数据 (对应原本的第二个cell k60100) ====================
files_right = [
    "/home/wuct/MetaData/DATA/OO/apass2/corr/results/fifth/k80100/k80100/CorrelExtract_0d_1d3_AppDeltaPhi/CorrelationsResults/CorrelationsResults.root",
    "/home/wuct/MetaData/DATA/OO/apass2/corr/results/fifth/k80100/k80100/CorrelExtract_0d2_1d3_AppDeltaPhi/CorrelationsResults/CorrelationsResults.root",
    "/home/wuct/MetaData/DATA/OO/apass2/corr/results/fifth/k80100/k80100/CorrelExtract_0d4_1d3_AppDeltaPhi/CorrelationsResults/CorrelationsResults.root",
    "/home/wuct/MetaData/DATA/OO/apass2/corr/results/fifth/k80100/k80100/CorrelExtract_0d4_1d3_AppDeltaPhi/CorrelationsResults/CorrelationsResults.root",
    "/home/wuct/MetaData/DATA/OO/apass2/corr/results/fifth/k80100/k80100/CorrelExtract_0d8_1d3_AppDeltaPhi/CorrelationsResults/CorrelationsResults.root",
]
objNames_right = [
    "PtCandBin_20_25/PtHadBin_2_30/DeltaPhiBin_0_392/hNormalizedCorrectedCorrel",
    "PtCandBin_20_25/PtHadBin_2_30/DeltaPhiBin_0_392/hNormalizedCorrectedCorrel",
    "PtCandBin_20_25/PtHadBin_2_30/DeltaPhiBin_0_392/hNormalizedCorrectedCorrel",
    "PtCandBin_20_25/PtHadBin_2_30/DeltaPhiBin_0_392/hNormalizedCorrectedCorrel",
    "PtCandBin_20_25/PtHadBin_2_30/DeltaPhiBin_0_392/hNormalizedCorrectedCorrel"
]
legendTexts_right = [
    "Corrected SE 80-100 0.0-1.3 LM",
    "Corrected SE 80-100 0.2-1.3 LM",
    "Corrected SE 80-100 0.4-1.3 LM",
    "Corrected SE 80-100 0.6-1.3 LM",
    "Corrected SE 80-100 0.8-1.3 LM",
]

ptBinsCand = [0, 1.0, 1.5, 2.0, 2.5, 3.0, 3.5, 4.0, 5.0, 6.0, 8.0, 12.0]
ptMins = ptBinsCand[:-1]
ptMaxs = ptBinsCand[1:]
pt_objNames_left = []
pt_objNames_right = []
for ptMin, ptMax in zip(ptMins, ptMaxs):
    ptBinStr = f"PtCandBin_{int(ptMin*10)}_{int(ptMax*10)}"
    ptLabel = f"{ptMin:.1f} < p_{{T}}^{{trigger}} < {ptMax:.1f} GeV/c"
    pt_objNames_left = [name.replace("PtCandBin_50_60", ptBinStr) for name in objNames_left]
    pt_objNames_right = [name.replace("PtCandBin_20_25", ptBinStr) for name in objNames_right]
    suffix = f"CorrectedSE_7080_AppDeltaPhi_Pt{int(ptMin*10)}_{int(ptMax*10)}"
    compare_se_combined(
        files_left, pt_objNames_left, legendTexts_left, 
        files_right, pt_objNames_right, legendTexts_right, 
        outputPath, suffix, ptLabel
    )
# 统一输出文件后缀
# suffix = "CorrectedSE_AppDeltaPhi_Combined"

# compare_se_combined(
#     files_left, objNames_left, legendTexts_left, 
#     files_right, objNames_right, legendTexts_right, 
#     outputPath, suffix
# )

Warning in <TCanvas::Constructor>: Deleting canvas with same name: c1
Info in <TCanvas::Print>: png file /home/wuct/MetaData/DATA/OO/apass2/corr/results/fifth/k020/etavariation/compare/CorrectedSE/compare_CorrectedSE_AppDeltaPhi_Pt0_10.png has been created
Info in <TCanvas::Print>: png file /home/wuct/MetaData/DATA/OO/apass2/corr/results/fifth/k020/etavariation/compare/CorrectedSE/compare_CorrectedSE_AppDeltaPhi_Pt10_15.png has been created
Info in <TCanvas::Print>: png file /home/wuct/MetaData/DATA/OO/apass2/corr/results/fifth/k020/etavariation/compare/CorrectedSE/compare_CorrectedSE_AppDeltaPhi_Pt15_20.png has been created
Info in <TCanvas::Print>: png file /home/wuct/MetaData/DATA/OO/apass2/corr/results/fifth/k020/etavariation/compare/CorrectedSE/compare_CorrectedSE_AppDeltaPhi_Pt20_25.png has been created
Info in <TCanvas::Print>: png file /home/wuct/MetaData/DATA/OO/apass2/corr/results/fifth/k020/etavariation/compare/CorrectedSE/compare_CorrectedSE_AppDeltaPhi_Pt25_30.png has been

In [74]:
import os
import ROOT

def compare_se_hm_lm_ratio(files_lm, objNames_lm, legendTexts_lm, 
                           files_hm, objNames_hm, legendTexts_hm, 
                           outputPath, suffix, pt_range=""):
    colors = [
        ROOT.kRed-4, ROOT.kBlue-4, ROOT.kGreen+2, ROOT.kOrange+7,
        ROOT.kMagenta-3, ROOT.kCyan-3, ROOT.kSpring-5, ROOT.kViolet-4,
        ROOT.kTeal-5, ROOT.kGray+1
    ]
    markerStyles = [20, 20, 20, 20, 20, 21, 21, 21, 21, 22]

    # 辅函数：负责读取文件、处理直方图并平移最小值
    def process_histograms(files, objNames, pad_id):
        objs = []
        for i, (fpath, objName, color, markerStyle) in enumerate(zip(files, objNames, colors, markerStyles)):
            f = ROOT.TFile.Open(fpath, "read")
            obj = f.Get(objName)
            # obj.Rebin(2)
            
            if isinstance(obj, ROOT.TH2):
                obj = obj.ProjectionY(f"proj_{pad_id}_{i}")
                temp_min = obj.GetMinimum()
            elif isinstance(obj, ROOT.TH1):
                temp_min = obj.GetMinimum()
                temp = obj.Clone(f"temp_{pad_id}_{i}")
                temp.Reset()
                for b in range(1, obj.GetNbinsX() + 1):
                    temp.SetBinContent(b, -temp_min)
                # obj.Add(temp)
            else:
                print(f"Object {objName} in file {fpath} is neither TH1 nor TH2. Skipping.")
                continue
                
            obj.SetDirectory(0) if hasattr(obj, "SetDirectory") else None
            obj.SetLineColor(color)
            obj.SetLineWidth(3)
            obj.SetMarkerColor(color)
            obj.SetMarkerStyle(markerStyle)
            obj.SetMarkerSize(1)
            
            # 最后一个直方图设为黑色 (如果不需要可以注释掉)
            if i == len(files) - 1:
                obj.SetLineColor(ROOT.kBlack)
                obj.SetMarkerColor(ROOT.kBlack)
                obj.SetMarkerStyle(20)
                
            obj.temp_min = temp_min # 保存平移的数值供Legend使用
            objs.append(obj)
            f.Close()
        return objs

    # 分别获取 LM 和 HM 的直方图对象
    objs_lm = process_histograms(files_lm, objNames_lm, "lm")
    objs_hm = process_histograms(files_hm, objNames_hm, "hm")

    # ================= 准备画布 =================
    canvas = ROOT.TCanvas("c1", "c1", 3600, 1500) 
    canvas.Divide(2, 1)

    # ================= 左图: 画 LM 分布 =================
    canvas.cd(1)
    ROOT.gPad.SetLeftMargin(0.15)
    ROOT.gPad.SetBottomMargin(0.15)
    ROOT.gPad.SetRightMargin(0.05)
    ROOT.gPad.SetTopMargin(0.08)
    ROOT.gStyle.SetOptStat(0)

    Legend_lm = ROOT.TLegend(0.2, 0.7, 0.6, 0.9)
    Legend_lm.SetBorderSize(0)
    Legend_lm.SetFillStyle(0)
    Legend_lm.SetTextFont(42)  
    Legend_lm.SetTextSize(0.04)

    max_y_lm = max([obj.GetMaximum() for obj in objs_lm]) if objs_lm else 0
    global_max_y_lm = max_y_lm * 1.5

    for i, obj in enumerate(objs_lm):
        if i == 0:
            obj.GetYaxis().SetTitle("1/dN^{trigger} #frac{dN}{d#Delta#varphi}")
            obj.GetYaxis().SetRangeUser(0, global_max_y_lm)
            obj.GetXaxis().SetTitle("#Delta#varphi (rad)")
            obj.GetXaxis().SetTitleOffset(1.2)
            obj.SetTitle(pt_range + " (LM)")
            obj.Draw("same pe")
        else:
            obj.Draw("same pe")
        Legend_lm.AddEntry(obj, f"{legendTexts_lm[i]}", "lp")
    Legend_lm.Draw()

    # ================= 右图: 画 HM / LM 的 Ratio =================
    canvas.cd(2)
    ROOT.gPad.SetLeftMargin(0.15)
    ROOT.gPad.SetBottomMargin(0.15)
    ROOT.gPad.SetRightMargin(0.05)
    ROOT.gPad.SetTopMargin(0.08)

    Legend_ratio = ROOT.TLegend(0.2, 0.7, 0.6, 0.9)
    Legend_ratio.SetBorderSize(0)
    Legend_ratio.SetFillStyle(0)
    Legend_ratio.SetTextFont(42)  
    Legend_ratio.SetTextSize(0.04)

    ratio_objs = []
    # 遍历计算 HM/LM，假定两者文件数量一一对应
    for i in range(min(len(objs_hm), len(objs_lm))):
        ratio = objs_hm[i].Clone(f"ratio_hm_lm_{i}")
        ratio.Divide(objs_lm[i])
        ratio.SetLineColor(objs_hm[i].GetLineColor())
        ratio.SetLineWidth(3)
        ratio.SetMarkerColor(objs_hm[i].GetMarkerColor())
        ratio.SetMarkerStyle(objs_hm[i].GetMarkerStyle())
        ratio.SetMarkerSize(1)
        ratio.GetYaxis().SetTitle("Ratio (HM / LM)")
        ratio.GetXaxis().SetTitle("#Delta#varphi (rad)")
        ratio.GetXaxis().SetTitleOffset(1.2)
        ratio.SetTitle(pt_range + " Ratio")
        ratio_objs.append(ratio)

    if ratio_objs:
        # 动态计算 Ratio 的 Y 轴范围
        max_ratios = [r.GetMaximum() for r in ratio_objs]
        min_ratios = [r.GetMinimum() for r in ratio_objs]
        y_max_ratio = max(max_ratios) * 1.3 if max(max_ratios) > 0 else 2
        y_min_ratio = min(min_ratios) * 0.7 if min(min_ratios) > 0 else 0

        for i, ratio in enumerate(ratio_objs):
            if i == 0:
                ratio.GetYaxis().SetRangeUser(y_min_ratio, y_max_ratio)
                ratio.Draw("same pe")
            else:
                ratio.Draw("same pe")
            # 图例标注为对应设定的名称 HM / LM
            Legend_ratio.AddEntry(ratio, f"{legendTexts_hm[i].replace(' LM', '')} HM/LM", "lp")
            
    Legend_ratio.Draw()

    # 在 Ratio 图中加一条 y=1 的基准参考线
    line = ROOT.TLine(-1.58, 1, 4.72, 1)
    line.SetLineStyle(2)
    line.SetLineColor(ROOT.kBlack)
    line.Draw("same")

    canvas.Update()

    # ================= 保存文件 =================
    os.system(f"mkdir -p {outputPath}")
    output_file_path = os.path.join(outputPath, f"compare_{suffix}.root")
    canvas.SaveAs(os.path.join(outputPath, f"compare_{suffix}.png"))
    
    outputfile = ROOT.TFile(output_file_path, "recreate")
    canvas.Write("c1")
    for obj in objs_lm + objs_hm + ratio_objs:
        obj.Write()
    outputfile.Close()

In [ ]:
outputPath = "/home/wuct/MetaData/DATA/OO/apass2/corr/results/fifth/k020/etavariation/compare/CorrectedSE/k70100/"

# ==================== 左图数据 (对应原本的第三个cell k020) ====================
files_left = [
    "/home/wuct/MetaData/DATA/OO/apass2/corr/results/fifth/k020/etavariation/CorrelExtract_0d_1d3_Appmass/CorrelationsResults/CorrelationsResults.root",
    "/home/wuct/MetaData/DATA/OO/apass2/corr/results/fifth/k70100/k70100/CorrelExtract_0d2_1d3_AppDeltaPhi/CorrelationsResults/CorrelationsResults.root",
    "/home/wuct/MetaData/DATA/OO/apass2/corr/results/fifth/k70100/k70100/CorrelExtract_0d4_1d3_AppDeltaPhi/CorrelationsResults/CorrelationsResults.root",
    "/home/wuct/MetaData/DATA/OO/apass2/corr/results/fifth/k70100/k70100/CorrelExtract_0d6_1d3_AppDeltaPhi/CorrelationsResults/CorrelationsResults.root",
    "/home/wuct/MetaData/DATA/OO/apass2/corr/results/fifth/k70100/k70100/CorrelExtract_0d8_1d3_AppDeltaPhi/CorrelationsResults/CorrelationsResults.root",
]
objNames_left = [
    "PtCandBin_50_60/PtHadBin_2_30/InvMassBin_1720_1760/hNormalizedCorrectedCorrel",
    "PtCandBin_50_60/PtHadBin_2_30/DeltaPhiBin_0_392/hNormalizedCorrectedCorrel",
    "PtCandBin_50_60/PtHadBin_2_30/DeltaPhiBin_0_392/hNormalizedCorrectedCorrel",
    "PtCandBin_50_60/PtHadBin_2_30/DeltaPhiBin_0_392/hNormalizedCorrectedCorrel",
    "PtCandBin_50_60/PtHadBin_2_30/DeltaPhiBin_0_392/hNormalizedCorrectedCorrel"
]
legendTexts_left = [
    "Corrected SE 0.0-1.3 LM 70-100",
    "Corrected SE 0.2-1.3 LM 70-100",
    "Corrected SE 0.4-1.3 LM 70-100",
    "Corrected SE 0.6-1.3 LM 70-100",
    "Corrected SE 0.8-1.3 LM 70-100",
]

# ==================== 右图数据 (对应原本的第二个cell k60100) ====================
files_right = [
    "/home/wuct/MetaData/DATA/OO/apass2/corr/results/fifth/k020/etavariation/CorrelExtract_0d_1d3_AppDeltaPhi/CorrelationsResults/CorrelationsResults.root",
    "/home/wuct/MetaData/DATA/OO/apass2/corr/results/fifth/k020/etavariation/CorrelExtract_0d2_1d3_AppDeltaPhi/CorrelationsResults/CorrelationsResults.root",
    "/home/wuct/MetaData/DATA/OO/apass2/corr/results/fifth/k020/etavariation/CorrelExtract_0d4_1d3_AppDeltaPhi/CorrelationsResults/CorrelationsResults.root",
    "/home/wuct/MetaData/DATA/OO/apass2/corr/results/fifth/k020/etavariation/CorrelExtract_0d6_1d3_AppDeltaPhi/CorrelationsResults/CorrelationsResults.root",
    "/home/wuct/MetaData/DATA/OO/apass2/corr/results/fifth/k020/fitprocedure/CorrelExtract_0d8_1d3_AppDeltaPhi/CorrelationsResults/CorrelationsResults.root",
]
objNames_right = [
    "PtCandBin_20_25/PtHadBin_2_30/DeltaPhiBin_0_392/hNormalizedCorrectedCorrel",
    "PtCandBin_20_25/PtHadBin_2_30/DeltaPhiBin_0_392/hNormalizedCorrectedCorrel",
    "PtCandBin_20_25/PtHadBin_2_30/DeltaPhiBin_0_392/hNormalizedCorrectedCorrel",
    "PtCandBin_20_25/PtHadBin_2_30/DeltaPhiBin_0_392/hNormalizedCorrectedCorrel",
    "PtCandBin_20_25/PtHadBin_2_30/DeltaPhiBin_0_392/hNormalizedCorrectedCorrel"
]
legendTexts_right = [
    "Corrected SE 0.0-1.3",
    "Corrected SE 0.2-1.3",
    "Corrected SE 0.4-1.3",
    "Corrected SE 0.6-1.3",
    "Corrected SE 0.8-1.3",
]

ptBinsCand = [0, 1.0, 1.5, 2.0, 2.5, 3.0, 3.5, 4.0, 5.0, 6.0, 8.0, 12.0]
ptMins = ptBinsCand[:-1]
ptMaxs = ptBinsCand[1:]
pt_objNames_left = []
pt_objNames_right = []
for ptMin, ptMax in zip(ptMins, ptMaxs):
    ptBinStr = f"PtCandBin_{int(ptMin*10)}_{int(ptMax*10)}"
    ptLabel = f"{ptMin:.1f} < p_{{T}}^{{trigger}} < {ptMax:.1f} GeV/c"
    pt_objNames_left = [name.replace("PtCandBin_50_60", ptBinStr) for name in objNames_left]
    pt_objNames_right = [name.replace("PtCandBin_20_25", ptBinStr) for name in objNames_right]
    suffix = f"CorrectedSE_AppDeltaPhi_Pt{int(ptMin*10)}_{int(ptMax*10)}"
    compare_se_hm_lm_ratio(
        files_left, pt_objNames_left, legendTexts_left, 
        files_right, pt_objNames_right, legendTexts_right, 
        outputPath, suffix, ptLabel
    )
# 统一输出文件后缀
# suffix = "CorrectedSE_AppDeltaPhi_Combined"

# compare_se_combined(
#     files_left, objNames_left, legendTexts_left, 
#     files_right, objNames_right, legendTexts_right, 
#     outputPath, suffix
# )

Info in <TCanvas::Print>: png file /home/wuct/MetaData/DATA/OO/apass2/corr/results/fifth/k020/etavariation/compare/CorrectedSE/k70100/compare_CorrectedSE_AppDeltaPhi_Pt0_10.png has been created
Info in <TCanvas::Print>: png file /home/wuct/MetaData/DATA/OO/apass2/corr/results/fifth/k020/etavariation/compare/CorrectedSE/k70100/compare_CorrectedSE_AppDeltaPhi_Pt10_15.png has been created
Info in <TCanvas::Print>: png file /home/wuct/MetaData/DATA/OO/apass2/corr/results/fifth/k020/etavariation/compare/CorrectedSE/k70100/compare_CorrectedSE_AppDeltaPhi_Pt15_20.png has been created
Info in <TCanvas::Print>: png file /home/wuct/MetaData/DATA/OO/apass2/corr/results/fifth/k020/etavariation/compare/CorrectedSE/k70100/compare_CorrectedSE_AppDeltaPhi_Pt20_25.png has been created
Info in <TCanvas::Print>: png file /home/wuct/MetaData/DATA/OO/apass2/corr/results/fifth/k020/etavariation/compare/CorrectedSE/k70100/compare_CorrectedSE_AppDeltaPhi_Pt25_30.png has been created
Info in <TCanvas::Print>: 

In [76]:
outputPath = "/home/wuct/MetaData/DATA/OO/apass2/corr/results/fifth/k020/etavariation/compare/CorrectedSE/k80100/"

# ==================== 左图数据 (对应原本的第三个cell k020) ====================
files_left = [
    "/home/wuct/MetaData/DATA/OO/apass2/corr/results/fifth/k80100/k80100/CorrelExtract_0d_1d3_AppDeltaPhi/CorrelationsResults/CorrelationsResults.root",
    "/home/wuct/MetaData/DATA/OO/apass2/corr/results/fifth/k80100/k80100/CorrelExtract_0d2_1d3_AppDeltaPhi/CorrelationsResults/CorrelationsResults.root",
    "/home/wuct/MetaData/DATA/OO/apass2/corr/results/fifth/k80100/k80100/CorrelExtract_0d4_1d3_AppDeltaPhi/CorrelationsResults/CorrelationsResults.root",
    "/home/wuct/MetaData/DATA/OO/apass2/corr/results/fifth/k80100/k80100/CorrelExtract_0d6_1d3_AppDeltaPhi/CorrelationsResults/CorrelationsResults.root",
    "/home/wuct/MetaData/DATA/OO/apass2/corr/results/fifth/k80100/k80100/CorrelExtract_0d8_1d3_AppDeltaPhi/CorrelationsResults/CorrelationsResults.root",
]
objNames_left = [
    "PtCandBin_50_60/PtHadBin_2_30/DeltaPhiBin_0_392/hNormalizedCorrectedCorrel",
    "PtCandBin_50_60/PtHadBin_2_30/DeltaPhiBin_0_392/hNormalizedCorrectedCorrel",
    "PtCandBin_50_60/PtHadBin_2_30/DeltaPhiBin_0_392/hNormalizedCorrectedCorrel",
    "PtCandBin_50_60/PtHadBin_2_30/DeltaPhiBin_0_392/hNormalizedCorrectedCorrel",
    "PtCandBin_50_60/PtHadBin_2_30/DeltaPhiBin_0_392/hNormalizedCorrectedCorrel"
]
legendTexts_left = [
    "Corrected SE 0.0-1.3 LM 70-100",
    "Corrected SE 0.2-1.3 LM 70-100",
    "Corrected SE 0.4-1.3 LM 70-100",
    "Corrected SE 0.6-1.3 LM 70-100",
    "Corrected SE 0.8-1.3 LM 70-100",
]

# ==================== 右图数据 (对应原本的第二个cell k60100) ====================
files_right = [
    "/home/wuct/MetaData/DATA/OO/apass2/corr/results/fifth/k020/etavariation/CorrelExtract_0d_1d3_AppDeltaPhi/CorrelationsResults/CorrelationsResults.root",
    "/home/wuct/MetaData/DATA/OO/apass2/corr/results/fifth/k020/etavariation/CorrelExtract_0d2_1d3_AppDeltaPhi/CorrelationsResults/CorrelationsResults.root",
    "/home/wuct/MetaData/DATA/OO/apass2/corr/results/fifth/k020/etavariation/CorrelExtract_0d4_1d3_AppDeltaPhi/CorrelationsResults/CorrelationsResults.root",
    "/home/wuct/MetaData/DATA/OO/apass2/corr/results/fifth/k020/etavariation/CorrelExtract_0d6_1d3_AppDeltaPhi/CorrelationsResults/CorrelationsResults.root",
    "/home/wuct/MetaData/DATA/OO/apass2/corr/results/fifth/k020/fitprocedure/CorrelExtract_0d8_1d3_AppDeltaPhi/CorrelationsResults/CorrelationsResults.root",
]
objNames_right = [
    "PtCandBin_20_25/PtHadBin_2_30/DeltaPhiBin_0_392/hNormalizedCorrectedCorrel",
    "PtCandBin_20_25/PtHadBin_2_30/DeltaPhiBin_0_392/hNormalizedCorrectedCorrel",
    "PtCandBin_20_25/PtHadBin_2_30/DeltaPhiBin_0_392/hNormalizedCorrectedCorrel",
    "PtCandBin_20_25/PtHadBin_2_30/DeltaPhiBin_0_392/hNormalizedCorrectedCorrel",
    "PtCandBin_20_25/PtHadBin_2_30/DeltaPhiBin_0_392/hNormalizedCorrectedCorrel"
]
legendTexts_right = [
    "Corrected SE 0.0-1.3",
    "Corrected SE 0.2-1.3",
    "Corrected SE 0.4-1.3",
    "Corrected SE 0.6-1.3",
    "Corrected SE 0.8-1.3",
]

ptBinsCand = [0, 1.0, 1.5, 2.0, 2.5, 3.0, 3.5, 4.0, 5.0, 6.0, 8.0, 12.0]
ptMins = ptBinsCand[:-1]
ptMaxs = ptBinsCand[1:]
pt_objNames_left = []
pt_objNames_right = []
for ptMin, ptMax in zip(ptMins, ptMaxs):
    ptBinStr = f"PtCandBin_{int(ptMin*10)}_{int(ptMax*10)}"
    ptLabel = f"{ptMin:.1f} < p_{{T}}^{{trigger}} < {ptMax:.1f} GeV/c"
    pt_objNames_left = [name.replace("PtCandBin_50_60", ptBinStr) for name in objNames_left]
    pt_objNames_right = [name.replace("PtCandBin_20_25", ptBinStr) for name in objNames_right]
    suffix = f"CorrectedSE_AppDeltaPhi_Pt{int(ptMin*10)}_{int(ptMax*10)}"
    compare_se_hm_lm_ratio(
        files_left, pt_objNames_left, legendTexts_left, 
        files_right, pt_objNames_right, legendTexts_right, 
        outputPath, suffix, ptLabel
    )
# 统一输出文件后缀
# suffix = "CorrectedSE_AppDeltaPhi_Combined"

# compare_se_combined(
#     files_left, objNames_left, legendTexts_left, 
#     files_right, objNames_right, legendTexts_right, 
#     outputPath, suffix
# )

Info in <TCanvas::Print>: png file /home/wuct/MetaData/DATA/OO/apass2/corr/results/fifth/k020/etavariation/compare/CorrectedSE/k80100/compare_CorrectedSE_AppDeltaPhi_Pt0_10.png has been created
Info in <TCanvas::Print>: png file /home/wuct/MetaData/DATA/OO/apass2/corr/results/fifth/k020/etavariation/compare/CorrectedSE/k80100/compare_CorrectedSE_AppDeltaPhi_Pt10_15.png has been created
Info in <TCanvas::Print>: png file /home/wuct/MetaData/DATA/OO/apass2/corr/results/fifth/k020/etavariation/compare/CorrectedSE/k80100/compare_CorrectedSE_AppDeltaPhi_Pt15_20.png has been created
Info in <TCanvas::Print>: png file /home/wuct/MetaData/DATA/OO/apass2/corr/results/fifth/k020/etavariation/compare/CorrectedSE/k80100/compare_CorrectedSE_AppDeltaPhi_Pt20_25.png has been created
Info in <TCanvas::Print>: png file /home/wuct/MetaData/DATA/OO/apass2/corr/results/fifth/k020/etavariation/compare/CorrectedSE/k80100/compare_CorrectedSE_AppDeltaPhi_Pt25_30.png has been created
Info in <TCanvas::Print>: 